# Fortgeschrittene Rekursionsmuster in UnifyWeaver

Dieses Notebook demonstriert die vier wichtigsten Rekursionsmuster, die UnifyWeaver erkennen und optimieren kann:

1. **Endrekursion (Tail Recursion)** - Iterative Schleifen mit Akkumulatoren
2. **Lineare Rekursion (Linear Recursion)** - Einzelner rekursiver Aufruf mit Memoisierung
3. **Baumrekursion (Tree Recursion)** - Mehrere rekursive Aufrufe auf Strukturteilen
4. **Wechselseitige Rekursion (Mutual Recursion)** - Prädikate, die sich zyklisch gegenseitig aufrufen

## Lernziele

- Verschiedene Rekursionsmuster verstehen
- Sehen, wie UnifyWeaver jedes Muster erkennt und optimiert
- Leistungsmerkmale vergleichen
- Lernen, wann welches Muster verwendet werden sollte

## Einrichtung

Initialisieren der UnifyWeaver-Umgebung.

In [ ]:
% Initialisierung laden
['../init'].

% Erforderliche Module laden
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## Muster 1: Endrekursion (Tail Recursion)

Endrekursion verwendet einen Akkumulator zur Übergabe von Zwischenergebnissen, und der rekursive Aufruf ist die **letzte Aktion** in der Funktion.

### Beispiel: Elemente in einer Liste zählen

In [ ]:
% Endrekursives count_items definieren
:- dynamic count_items/3.

% Basisfall: Leere Liste, Akkumulator zurückgeben
count_items([], Acc, Acc).

% Rekursiver Fall: Akkumulator erhöhen, Rekursion auf der Restliste
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← Endrekursive Position!

### In Prolog testen

In [ ]:
% Test: Elemente in [a,b,c,d,e] zählen
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### Mustererkennung prüfen

In [ ]:
% Prüfen, ob als endrekursiv erkannt
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### Nach Bash kompilieren

In [ ]:
% Kompilieren und speichern
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### Generiertes Bash testen

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "Zähle Elemente in [a,b,c,d,e]:"
count_items "[a,b,c,d,e]" 0 ""

## Muster 2: Lineare Rekursion (Linear Recursion)

Lineare Rekursion besitzt **genau einen** rekursiven Aufruf pro Klausel, wobei Berechnungen nach der Rückkehr des rekursiven Aufrufs stattfinden.

### Beispiel: Fakultät (Factorial)

In [ ]:
% Fakultät definieren
:- dynamic factorial/2.

% Basisfall
factorial(0, 1).

% Rekursiver Fall: Genau EIN rekursiver Aufruf
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← Ein rekursiver Aufruf
    F is N * F1.        % ← Berechnung nach dem Aufruf

### In Prolog testen

In [ ]:
% Test: Fakultät von 5
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### Mustererkennung prüfen

In [ ]:
% Prüfen, ob als linear rekursiv erkannt
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### Nach Bash kompilieren

In [ ]:
% Kompilieren und speichern
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % Nur Funktionsdefinitionen behalten; Brush behandelt gesourcte Skripte als direkte Ausführung
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### Generiertes Bash testen

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "Fakultät von 5:"
factorial 5 ""
echo ""
echo "Fakultät von 10:"
factorial 10 ""

## Muster 3: Baumrekursion (Tree Recursion)

Baumrekursion führt **mehrere** rekursive Aufrufe aus, um verschiedene Teile einer Struktur zu verarbeiten.

### Beispiel: Baumsumme (Tree Sum)

In [ ]:
% tree_sum für Binärbäume definieren
% Baumformat: [Wert, LinkerTeilbaum, RechterTeilbaum] oder []
:- dynamic tree_sum/2.

% Basisfall: Leerer Baum hat Summe 0
tree_sum([], 0).

% Rekursiver Fall: Summe = Wert + linke_Summe + rechte_Summe
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← Erster rekursiver Aufruf
    tree_sum(R, RS),   % ← Zweiter rekursiver Aufruf
    Sum is V + LS + RS.

### In Prolog testen

In [ ]:
% Test: tree_sum von [5, [3, [1, [], []], []], [2, [], []]]
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### Nach Bash kompilieren

In [ ]:
% Kompilieren und speichern
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### Generiertes Bash testen

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "Baumsumme von [5,[3,[1,[],[]],[]],[2,[],[]]]:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## Muster 4: Wechselseitige Rekursion (Mutual Recursion)

Wechselseitige Rekursion tritt auf, wenn zwei oder mehr Prädikate sich gegenseitig in einem Zyklus aufrufen.

### Beispiel: Gerade (Even) und Ungerade (Odd)

In [ ]:
% Wechselseitig rekursive Prädikate is_even und is_odd definieren
:- dynamic is_even/1.
:- dynamic is_odd/1.

% is_even Basisfall
is_even(0).

% is_even rekursiv: N ist gerade, wenn N-1 ungerade ist
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← Ruft is_odd auf

% is_odd Basisfall
is_odd(1).

% is_odd rekursiv: N ist ungerade, wenn N-1 gerade ist
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← Ruft is_even auf

### In Prolog testen

In [ ]:
% Gerade/Ungerade testen
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### Auf wechselseitige Rekursion prüfen

In [ ]:
% Aufrufgraph erstellen und SCCs finden
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### Nach Bash kompilieren

In [ ]:
% Gruppe mit wechselseitiger Rekursion kompilieren
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### Generiertes Bash testen

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "is_even und is_odd testen:"
is_even 0 >/dev/null && echo "✓ 0 ist gerade"
is_even 4 >/dev/null && echo "✓ 4 ist gerade"
is_odd 3 >/dev/null && echo "✓ 3 ist ungerade"
is_odd 7 >/dev/null && echo "✓ 7 ist ungerade"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 ist nicht gerade"

## Mustervergleich

Vergleichen wir die Eigenschaften der einzelnen Muster:

| Muster | Rekursive Aufrufe | Optimierung | Platzkomplexität | Am besten geeignet für |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **Endrekursion** | 1 (in Endposition) | Iterative Schleife | O(1) | Akkumulatoren, lineare Scans |
| **Lineare Rekursion** | 1 (beliebige Position) | Fold + Memoisierung | O(n) Memo-Tabelle | Fibonacci, Fakultät |
| **Baumrekursion** | 2+ (Strukturteile) | Strukturelle Zerlegung | O(Tiefe) Stack | Baum-/Graphoperationen |
| **Wechselseitig** | 1+ (prädikatsübergreifend) | Geteilte Memoisierung | O(n) geteilte Tabelle | Gerade/Ungerade, gegenseitige Definitionen |

## Reihenfolge der Mustererkennung

UnifyWeaver versucht den Musterabgleich in dieser Reihenfolge:

1. **Endrekursion** (am effizientesten)
2. **Lineare Rekursion** (sofern nicht verboten)
3. **Baumrekursion** (strukturell)
4. **Wechselseitige Rekursion** (SCC-Erkennung)
5. **Basis-Rekursion** (Fallback)

Sie können die Erkennung mit `forbid_linear_recursion/1` beeinflussen.

## Übung: Jetzt sind Sie dran!

Versuchen Sie, diese Prädikate zu definieren und zu kompilieren:

### 1. Endrekursive Summe
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. Linear-rekursive Fibonacci-Folge
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. Baumhöhe
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% Ihr Code hier!


## Zusammenfassung

In diesem Notebook haben Sie gelernt:

✅ Die vier wichtigsten Rekursionsmuster in UnifyWeaver

✅ Wie man jedes Muster in Prolog definiert

✅ Wie UnifyWeaver jedes Muster erkennt und optimiert

✅ Die Leistungsmerkmale jedes Musters

✅ Wann welches Muster verwendet werden sollte

## Nächste Schritte

Fahren Sie mit **Notebook 3: Visualisierung des Aufruf-Graphen** fort, um mehr über fortgeschrittene Code-Analyse und Visualisierung zu erfahren!